# Threat hunting playbook

Hypothesis-driven hunt on **labeled network-flow telemetry** and **published IOCs**, using Cribl Search [`externaldata`](https://docs.cribl.io/search/externaldata/) and **Search lookups**—no `pd.read_csv` in the browser for hunt data.

**Telemetry:** [CICIDS2017](https://unb.ca/cic/datasets/ids-2017.html) sample (Western-OC2-Lab mirror). **Intel:** [SophosLabs IoCs](https://github.com/sophoslabs/IoCs) (EDR-killer campaign, Aug 2025).

## Hunt charter

**Hypothesis:** A meaningful share of flows in the sample are attack-labeled (non-`BENIGN`), and published hash IOCs can be published to a Search lookup for enrichment-style hunting.

**Questions:**

1. What attack labels appear in the telemetry sample?
2. How do flow features differ between benign and attack-labeled rows?
3. Can we load third-party IOCs via `externaldata`, save them as a lookup, and query `$vt_lookups`?

**Scope:** Public CSVs only; lookup file `notebook_app_threat_hunt_iocs.csv` (deleted at the end).

## Prerequisites

1. Run this notebook **inside a hosted Cribl app** where **Cribl Search** works (same as `Anomaly_Detection_PyOD.ipynb` and `Cribl_Search_Lookup_Magics.ipynb`).
2. Search must reach `raw.githubusercontent.com` for `externaldata` URLs.
3. Lookups are capped at **10,000 rows**; this playbook uses a small IOC table (~50 rows).
4. If `datatype="CSV Datatypes"` fails on your tenant, change both `externaldata` cells to `datatype="CSV"` (see Anomaly notebook notes).

**Related examples:** `Cribl_Search_Lookup_Magics.ipynb` (lookup magics), `Anomaly_Detection_PyOD.ipynb` (`externaldata` pattern), `Incident_Triage_Playbook.ipynb` (IR-style triage on live sample data).

### Load telemetry (`externaldata` → `flows_df`)

Fetches the CICIDS2017 sample through Search (not Pyodide HTTP). We cap rows with `limit=5000` on the magic line. Filter non-benign labels in Python if `Label` is present.

In [ ]:
%%cribl_search var=flows_df lang=kql limit=5000 preview=true earliest=-50y latest=now
externaldata
[
  "https://raw.githubusercontent.com/Western-OC2-Lab/Intrusion-Detection-System-Using-Machine-Learning/main/data/CICIDS2017_sample.csv"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
import pandas as pd

print('flows rows:', len(flows_df))
print('columns (first 12):', list(flows_df.columns)[:12], '…' if len(flows_df.columns) > 12 else '')

_label_col = None
for c in flows_df.columns:
    if str(c).strip().lower() == 'label':
        _label_col = c
        break

if _label_col is None:
    print('No Label column yet — check Search CSV parsing or try datatype="CSV" in the cell above.')
    flows_hunt_df = flows_df.copy()
else:
    vc = flows_df[_label_col].astype(str).value_counts()
    print('Label distribution (top 10):')
    print(vc.head(10))
    flows_hunt_df = flows_df[flows_df[_label_col].astype(str).str.upper() != 'BENIGN'].copy()
    print('attack-labeled rows:', len(flows_hunt_df))

flows_hunt_df.head(5)

### Hunt pass — pivot attack labels and chart

Summarize attack-labeled flows and plot the top labels (Matplotlib is bundled with Pyodide; no `micropip` required).

In [ ]:
import matplotlib.pyplot as plt

label_col = next((c for c in flows_hunt_df.columns if str(c).strip().lower() == 'label'), None)

if label_col is None or len(flows_hunt_df) == 0:
    print('Skip chart: no attack-labeled rows or missing Label column.')
else:
    top = flows_hunt_df[label_col].astype(str).value_counts().head(10)
    print('Top attack labels:')
    print(top)
    ax = top.plot(kind='bar', title='Threat hunt: top attack labels (CICIDS sample)', rot=45)
    ax.set_xlabel('Label')
    ax.set_ylabel('Flow count')
    plt.tight_layout()
    plt.show()

### Load IOCs (`externaldata` → `ioc_raw`)

Published indicators from Sophos (SHA-256 and related types). Loaded through Search, then normalized for a lookup table.

In [ ]:
%%cribl_search var=ioc_raw lang=kql limit=0 preview=false earliest=-50y latest=now
externaldata
[
  "https://raw.githubusercontent.com/sophoslabs/IoCs/master/06082025-edrkiller-iocs.csv"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
def _pick_col(frame, *names):
    lower = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in lower:
            return lower[n]
    return None

tcol = _pick_col(ioc_raw, 'indicator_type')
dcol = _pick_col(ioc_raw, 'data')
ncol = _pick_col(ioc_raw, 'note')

if tcol and dcol:
    ioc_lookup_df = pd.DataFrame({
        'indicator_type': ioc_raw[tcol].astype(str),
        'value': ioc_raw[dcol].astype(str),
        'note': ioc_raw[ncol].astype(str) if ncol else '',
    })
    # Drop description/header rows that are not indicators
    ioc_lookup_df = ioc_lookup_df[
        ioc_lookup_df['indicator_type'].str.lower() != 'description'
    ].reset_index(drop=True)
else:
    # Fallback: first three columns
    cols = list(ioc_raw.columns)[:3]
    ioc_lookup_df = ioc_raw[cols].copy()
    ioc_lookup_df.columns = ['indicator_type', 'value', 'note'][: len(cols)]

print('IOC rows for lookup:', len(ioc_lookup_df))
ioc_lookup_df.head(10)

### Publish IOC lookup

Upload `ioc_lookup_df` to Search (`default_search` group). **`replace=true`** overwrites a prior run of this demo.

In [ ]:
%%cribl_save_search_lookup notebook_app_threat_hunt_iocs.csv var=ioc_lookup_df replace=true


### Hunt via `$vt_lookups`

Query the lookup through Search (`lookupFile` is the stem without `.csv`). See [Cribl `$vt_lookups` docs](https://docs.cribl.io/search/vt_lookups/).

In [ ]:
%%cribl_search var=vt_ioc_rows preview=true
dataset="$vt_lookups" lookupFile="notebook_app_threat_hunt_iocs"
| limit 25


In [ ]:
%%cribl_load_search_lookup notebook_app_threat_hunt_iocs.csv var=ioc_from_lookup


In [ ]:
print('flows (loaded):', len(flows_df))
print('flows (attack-labeled):', len(flows_hunt_df))
print('vt_lookups rows:', len(vt_ioc_rows))
print('lookup load rows:', len(ioc_from_lookup))
ioc_from_lookup.head(5)

### Production bridge

- Replace `externaldata` URLs with your own evidence exports or threat-intel feeds (HTTPS URLs Search can reach).
- On live data, run `dataset="your_index"` pipelines and **join** lookups in KQL instead of only analyzing CSV samples in pandas.
- This CICIDS sample has flow **features** and `Label`, not `src_ip`/`dst_ip`—use tenant telemetry for IP-centric hunts.
- Hash IOCs apply when you have endpoint or file-hash fields in your index; adapt lookup columns to match your schema.

In [ ]:
# ### Prompt:
# Draft a short threat-hunt summary from the notebook context below.
#
# Flow telemetry (attack-labeled sample):
# {{ flows_hunt_df | describe }}
#
# IOC lookup (first rows):
# {{ ioc_lookup_df | describe }}
#
# Requirements:
# - state whether the hypothesis is supported or inconclusive
# - list top 3 attack labels or patterns seen
# - note one limitation of CSV/externaldata-only hunting
# - suggest one next step on real tenant data (dataset= or lookup join)

### Cleanup

Removes the demo lookup `notebook_app_threat_hunt_iocs.csv` from the `default_search` group.

In [ ]:
%%cribl_delete_search_lookup notebook_app_threat_hunt_iocs.csv


## Debrief

- **Lookups:** `Cribl_Search_Lookup_Magics.ipynb`
- **`externaldata`:** `Anomaly_Detection_PyOD.ipynb`
- **Live-sample triage:** `Incident_Triage_Playbook.ipynb`

**Attribution:** CICIDS2017 ([UNB CIC](https://unb.ca/cic/datasets/ids-2017.html)); IOCs ([SophosLabs IoCs](https://github.com/sophoslabs/IoCs), EDR-killer report Aug 2025).